## 参考3.3生成的数据集，我们计划将随机生成的数据记录到文本文件中，方便发送给其他同学使用

将生成的数据集保存于csv文件中

In [2]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils import data
from d2l import torch as d2l

先生成并保存一份权重和偏置,文件名为answer_true.csv

In [3]:
#生成数据集w,b
true_w = torch.randn(100)
true_b = torch.randn(100)
#用于对照文件文本内容
print(true_w)
print(true_b)

tensor([ 0.6009,  0.2592, -1.7373,  0.3270, -1.6042,  1.0578,  1.3554,  0.2094,
         1.4017,  2.4915,  1.3800,  0.8557,  0.2115,  0.7389, -0.4312, -0.4622,
         0.5451, -0.6078, -1.3368,  0.7162,  1.4418, -1.6793, -0.0365, -1.4221,
         1.4319,  1.1939,  0.4989, -0.4190, -0.6044, -0.1445,  2.0940,  1.9070,
         0.1446, -0.0384, -0.0414, -2.3895,  1.6543, -2.2586,  1.0450, -1.4695,
        -1.0916, -0.6155,  1.7018, -0.9025,  2.0174, -0.5774,  0.6265, -0.0976,
         0.7033, -1.2493, -0.3821,  0.9778,  1.2386, -1.1320, -1.8507,  0.2928,
         0.5986, -0.3378, -0.8034,  0.2688,  1.0862, -1.5307,  0.9469,  1.1558,
         0.0470, -0.2730, -0.9068,  1.7283, -0.3434, -0.1364,  3.3323, -0.9662,
         0.3194, -1.1387, -1.8444, -2.1871,  0.1471, -0.6665, -0.2908,  0.8326,
         0.5704, -2.1707,  0.1943,  1.7506, -0.3517, -1.7490,  0.0880, -0.2497,
        -1.0903, -0.9791,  0.4523, -0.7672,  0.0360, -0.7160, -0.4113, -1.8000,
         0.0483, -0.3321,  1.5955, -0.68

```
TypeError                                 Traceback (most recent call last)
Cell In[4], line 5
      3 data_file = os.path.join('..', 'data', 'answer_true.csv')
      4 with open(data_file, 'w') as f:
----> 5     f.write('true_w','true_b')
      6     f=pd.DataFrame(true_w.numpy())
      7     f.to_csv('answer_true.csv',index=Fals)

TypeError: TextIOWrapper.write() takes exactly one argument (2 given)
```
文件对象的 write() 方法只能接收 1 个参数，但你传入了 2 个（或更多）。
write() 方法的设计规则是：一次只能写入单个字符串，不能像 print() 那样直接传多个参数（比如 f.write(a, b)）
```
AttributeError                            Traceback (most recent call last)
/tmp/ipykernel_4085/491788730.py in ?()
      6 data_file = os.path.join('..', 'data', 'answer_true.csv')
      7 with open(data_file, 'w') as f:
      8     f.write('true_w')
      9     f=pd.DataFrame(true_w.numpy())
---> 10     f.to_csv('answer_true.csv',index=False)
     11     f.write('true_b')
     12     f=pd.DataFrame(true_b.numpy())
     13     f.to_csv('answer_true.csv',index=False)

~/miniconda3/envs/d2l/lib/python3.9/site-packages/pandas/core/generic.py in ?(self, name)
   5461             return object.__getattribute__(self, name)
   5462         else:
   5463             if self._info_axis._can_hold_identifiers_and_holds_name(name):
   5464                 return self[name]
-> 5465             return object.__getattribute__(self, name)

AttributeError: 'DataFrame' object has no attribute 'write'
```
这个报错的核心原因是：你把变量 f 从 “文件对象” 覆盖成了 “DataFrame 对象”，而 Pandas 的 DataFrame 没有 write() 方法。
简单来说：
第 7 行：with open(...) as f → f 是文件写入对象（有 write() 方法）；
第 9 行：f=pd.DataFrame(...) → f 被替换成了 DataFrame（没有 write() 方法）；
第 11 行：f.write('true_b') → 用 DataFrame 调用 write()，直接触发 AttributeError。
同时你的代码还有两个隐藏问题：
to_csv 写入的路径是相对路径，会写到当前目录，而非你创建的 ../data/ 目录；
多次调用 to_csv 会覆盖文件内容，最终只保留最后一次写入的数据。


In [8]:
#exist_ok=True确保存在对应文件夹，不会报错结束任务执行
#确保路径存在
os.makedirs(os.path.join('..', 'data'), exist_ok=True)
#定义完整的文件路径（写到../data/下）
data_file = os.path.join('..', 'data', 'answer_true.csv')
# 正确写入：先构建DataFrame，再用to_csv写入（无需手动write）
# 将true_w和true_b合并写入同一个文件
df_w = pd.DataFrame(true_w.numpy(), columns=['true_w'])
df_b = pd.DataFrame(true_b.numpy(), columns=['true_b'])
# 合并两个DataFrame（按列合并）
#df_combined = pd.concat([df_w, df_b], axis=1) 的核心是：把两个 DataFrame 按 “列” 的方向拼接在一起（横向合并），形成一个新的 DataFrame。
#axis=1：按列拼接（横向，左右合并）axis=0（默认）：按行拼接（纵向，上下合并）
df_combined = pd.concat([df_w, df_b], axis=1)
# 写入到指定路径，不覆盖（mode='a'追加，首次写入用mode='w'）
df_combined.to_csv(data_file, index=False, mode='w')
answer = pd.read_csv(data_file)
print(answer)

      true_w    true_b
0   0.600903 -1.075110
1   0.259178 -0.703581
2  -1.737288  1.303858
3   0.327026  0.707368
4  -1.604199 -0.679120
..       ...       ...
95 -1.800013 -1.351284
96  0.048279  0.943185
97 -0.332105  0.236598
98  1.595545  1.176727
99 -0.688410 -0.550477

[100 rows x 2 columns]


### 接下来我们需要生成数据集并写入csv

由于用于测试，这里考虑将样本生成量调小为一万

In [9]:
def synthetic_data1(w, b, num_examples):  #@save
    """生成y=Xw+b+噪声"""
    X = torch.normal(0, 1, (num_examples, len(w)))
    #这里遇到xw与b无法广播，将生成num_examples行，每行为b的b_expanded
    b_expanded = b.repeat(num_examples // len(b) + 1)[:num_examples]
    Xw = torch.matmul(X, w) 
    y = Xw + b_expanded
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape((-1, 1))
features, labels = synthetic_data1(true_w, true_b, 10000)

这里遇到报错：
```
NameError                                 Traceback (most recent call last)
Cell In[10], line 4
      2 os.makedirs(os.path.join('..', 'data'), exist_ok=True)
      3 data_file = os.path.join('..', 'data', 'training_data.csv')
----> 4 df_X = pd.DataFrame(X.numpy(), columns=['X'])
      5 df_y = pd.DataFrame(y.numpy(), columns=['y'])
      6 df_combined = pd.concat([df_X, df_y], axis=1)

NameError: name 'X' is not defined
```
这里应该填入变量features, labels，而非函数中的X，y。

而且features是10000行100列的张量需考虑列设置问题

In [12]:
#将生成的X，y写入csv,这里应该填入变量features, labels，而非函数中的X，y。而且features是10000行100列的张量,需考虑列设置问题
os.makedirs(os.path.join('..', 'data'), exist_ok=True)
data_file = os.path.join('..', 'data', 'training_data.csv')
#创建列名：为100个特征生成列名列表
#f'feature_{i}':Python 的 f-string 格式化，把 i 拼接到字符串里
feature_cols = [f'feature_{i}' for i in range(features.shape[1])]  # ['feature_0', ..., 'feature_99']
#正确创建DataFrame（列名数量 = 数据列数）
df_features = pd.DataFrame(features.numpy(), columns=feature_cols)
df_labels = pd.DataFrame(labels.numpy(), columns=['labels'])
#横向拼接（特征+标签）
df_combined = pd.concat([df_features, df_labels], axis=1)

df_combined.to_csv(data_file, index=False, mode='w')
training = pd.read_csv(data_file)
print(training)

      feature_0  feature_1  feature_2  feature_3  feature_4  feature_5  \
0     -0.606976  -1.550677   0.396765  -0.365868   0.661423  -1.150686   
1      0.378828  -1.214507   1.823774  -0.313761   0.863215  -1.318341   
2     -0.463569  -0.225769  -0.296675  -0.984187   0.768370   1.825014   
3     -0.512921   0.924293   0.711535   0.521840   1.427171  -2.219130   
4      0.706100  -1.299183   0.064755  -0.024992   1.190404   0.347182   
...         ...        ...        ...        ...        ...        ...   
9995   0.792271   0.462823   0.202914  -0.371192   0.478116   1.549459   
9996   0.760748   0.263380  -0.553068   0.227103  -0.313271   0.414811   
9997  -0.480459   0.266423   3.171241   0.101727   1.555160   1.107533   
9998   1.373448  -0.800289   2.064278   0.434269   1.054408  -1.476384   
9999   0.386028   0.504425  -0.735255   1.108698  -0.296484  -1.959999   

      feature_6  feature_7  feature_8  feature_9  ...  feature_91  feature_92  \
0     -0.439694  -0.033389   1

### 小节：

这里成功生成了csv数据文本后续将推进读取并使用文本，完成3.3节的模型训练